# Домашнее задание 1. MLP против бустинга на табличных данных

ДЗ идёт после лекций 1–2 и семинаров к ним.

**Задача — Covertype.** По картографическим признакам участка леса в Колорадо
(высота, уклон, расстояния до воды и дорог, освещённость, тип почвы, заповедная
зона) предсказать, какой из **7 типов леса** на нём растёт. 581 012 участков.

**Соперник — градиентный бустинг.** На таблицах бустинг над деревьями обычно
сильнее полносвязной сети: ему всё равно, в каких шкалах признаки, и он почти
не требует настройки. На том же разбиении, что и у вас:

| Модель | Точность на тесте | Время обучения |
|---|---|---|
| CatBoost по умолчанию (1000 деревьев глубины 6) | 0,890 | ~10 мин на 8 ядрах |
| CatBoost посильнее (4000 деревьев глубины 8) | **0,960** | ~70 мин на 8 ядрах |

К концу ноутбука ваша сеть его обойдёт — за несколько минут обучения на видеокарте.
Covertype для сети удобный случай: полмиллиона строк и всего 54 признака.
На типичной таблице в несколько тысяч строк расклад обычно обратный.

## Как устроено задание

Почти весь код уже написан: загрузка данных, цикл обучения, графики.
Вам остаются **5 пропусков** по 3–12 строк (второй — в двух вариантах). Каждый — про механизм из лекций,
и каждый потом работает в настоящем обучении: учится **ваша** сеть на **ваших**
признаках, **вашей** функции потерь и **вашем** оптимизаторе. Пропуск выглядит так:

```python
def something(...):
    # TODO: ...
    raise NotImplementedError
```

**Сразу после каждого пропуска — ячейка проверки.** Она прогоняет ваш код
на маленьком искусственном примере и сравнивает с PyTorch.
Если что-то не так — упадёт с сообщением, что именно не сходится.
Если всё верно — напечатает `✅`. Ячейки проверки **не меняйте**.

| Часть | Пропуски | Порог на тесте | Оценка |
|---|---|---|---|
| 1. Своя сеть, которая учится (лекция 1) | 1–3 | ≥ 0,90 | «3» |
| 2. BatchNorm (лекция 2) | 4 | ≥ 0,95 | «4» |
| 3. Свой AdamW и финальная модель | 5 | ≥ 0,965 | «5» |

Оценка ставится за часть целиком: **все проверки части пройдены и тест
не ниже порога**. Части накопительные: «4» — это части 1 и 2, «5» — все три.
Итог подводит последняя ячейка ноутбука.

**Правила.**
* Не меняйте готовый код, разбиение данных, настройки обучения (`P1`, `P2`, `P3`)
  и ячейки проверки — пороги посчитаны ровно для них.
* В пропусках не используйте готовые реализации того же самого из PyTorch
  (`F.cross_entropy`, `F.log_softmax`, `torch.logsumexp`, `torch.optim` и т. п.) —
  с ними сверяются проверки.
* Сдаётся **этот ноутбук, выполненный целиком сверху вниз** («Перезапустить
  и выполнить всё»), с выводами ячеек.

**Как запускать.** Google Colab с видеокартой T4 (`Среда выполнения → Сменить
среду → T4 GPU`) или локально с CUDA; нужны `torch`, `scikit-learn`, `pandas`,
`matplotlib` — в Colab всё есть. Три обучения вместе — около 5–10 минут на T4.

In [ ]:
import math
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F

device = "cuda" if torch.cuda.is_available() else "cpu"
print("PyTorch:", torch.__version__, "| device:", device,
      "|", torch.cuda.get_device_name(0) if device == "cuda" else "")
if device == "cpu":
    print("⚠️ Видеокарты нет. В Colab: Среда выполнения → Сменить среду → T4 GPU.")

PASSED = {}          # сюда ячейки проверки записывают, что пройдено
TEST_ACC = {}        # сюда — точность на тесте по частям
T_START = time.time()


def passed(name, msg):
    PASSED[name] = True
    print("✅", msg)

---

## 0. Данные

`fetch_covtype` скачивает датасет один раз (~11 МБ) в папку `data/`.

* **10 числовых признаков**: высота (`Elevation`), азимут и уклон склона,
  расстояния до воды, дорог и мест пожаров, освещённость в 9, 12 и 15 часов;
* **`Wilderness_Area_0…3`** — одна из 4 заповедных зон, закодирована one-hot;
* **`Soil_Type_0…39`** — один из 40 типов почвы, тоже one-hot.

In [ ]:
from sklearn.datasets import fetch_covtype

cov = fetch_covtype(data_home="data")
X_all = cov.data.astype(np.float32)
y_all = (cov.target - 1).astype(np.int64)           # классы 1..7 → 0..6
NUM_NAMES = cov.feature_names[:10]

X_num_raw = X_all[:, :10]                           # 10 числовых признаков
X_onehot = X_all[:, 10:54]                          # 4 + 40 one-hot столбцов

print("объектов:", len(y_all), "| классов:", len(np.unique(y_all)))
print("доли классов:", np.round(np.bincount(y_all) / len(y_all), 3))
pd.DataFrame(X_num_raw, columns=NUM_NAMES).describe().T[["mean", "std", "min", "max"]].round(1)

Столбец `std`: у высоты около 280, у уклона 7,5, у расстояния до дорог — полторы
тысячи. Это та ситуация, ради которой на лекции была нормировка входов.
Классы несбалансированы: два самых частых дают 85% объектов, самый редкий — 0,5%;
ответ «всегда самый частый класс» даёт 0,488.

**Разбиение фиксировано** (70% train, 10% val, 20% test, стратифицировано по классу).
Не меняйте его: пороги посчитаны на этом тесте.

In [ ]:
from sklearn.model_selection import train_test_split

_idx = np.arange(len(y_all))
idx_trval, idx_test = train_test_split(_idx, test_size=0.2, random_state=42, stratify=y_all)
idx_train, idx_val = train_test_split(idx_trval, test_size=0.125, random_state=42,
                                      stratify=y_all[idx_trval])
print(f"train {len(idx_train)} | val {len(idx_val)} | test {len(idx_test)}")
print("контрольная сумма теста:", int(idx_test.sum()) % 1_000_003, "(должно быть 253923)")

---

# Часть 1. Сеть, которая учится → «3»

## Пропуск 1. Стандартизация входов

Каждый числовой признак приводим к нулевому среднему и единичному стандартному
отклонению: `(x − mean) / std`.

* `mean` и `std` считаются **только по обучающим строкам** `idx_train`, а применяются
  ко всем строкам. Если посчитать их по всей таблице, информация о валидации
  и тесте протечёт в обучение.
* Признак может оказаться постоянным на train (здесь таких нет, но функция
  должна это пережить): деление на ноль даст `inf` и `nan`. Такой столбец
  после стандартизации должен стать нулями.

Функция возвращает стандартизованную матрицу и сами статистики.

In [ ]:
def standardize(X, idx_train):
    """Стандартизация признаков по статистикам обучающих строк.

    X         — numpy-матрица признаков [N, d], все строки (train, val и test);
    idx_train — номера обучающих строк.
    Возвращает тройку (X_std, mean, std):
      X_std — стандартизованная матрица (X − mean) / std той же формы [N, d];
      mean, std — векторы длины d, посчитанные только по строкам idx_train.
    """
    # TODO 1: статистики по строкам idx_train, применить ко всем строкам X.
    #   Подсказка: np.mean / np.std с axis=0. Постоянный на train признак
    #   должен превратиться в нули, а не в nan.
    raise NotImplementedError

In [ ]:
# ПРОВЕРКА 1 — не меняйте эту ячейку
def _check_standardize():
    rng = np.random.RandomState(1)
    X = rng.randn(300, 3).astype(np.float32) * [2.0, 50.0, 1.0] + [5.0, -100.0, 0.0]
    X = X.astype(np.float32)
    tr = np.arange(0, 300, 3)                 # обучающие строки
    other = np.setdiff1d(np.arange(300), tr)
    X[other] += np.float32([40.0, 900.0, 3.0])  # остальные строки распределены совсем иначе
    X[tr, 2] = 7.0                            # третий признак на train постоянный
    Xs, mean, std = standardize(X.copy(), tr)
    Xs, mean = np.asarray(Xs, dtype=np.float64), np.asarray(mean, dtype=np.float64)
    assert Xs.shape == X.shape, f"форма результата {Xs.shape}, а должна быть {X.shape}"
    assert np.isfinite(Xs).all(), \
        "в результате есть nan или inf — постоянный на train признак не обработан"
    assert np.allclose(mean[:2], X[tr, :2].mean(0), rtol=1e-3), \
        "mean не совпадает со средним по обучающим строкам — посчитан не по тем строкам?"
    assert np.allclose(Xs[tr, :2].mean(0), 0, atol=1e-3), \
        "на обучающих строках среднее после стандартизации не 0"
    assert np.allclose(Xs[tr, :2].std(0), 1, atol=0.02), \
        "на обучающих строках std после стандартизации не 1"
    assert np.abs(Xs[other, 0].mean()) > 5, \
        "остальные строки тоже оказались с нулевым средним — статистики посчитаны по всей таблице"
    assert np.allclose(Xs[tr, 2], 0, atol=1e-6), \
        "постоянный на train признак должен стать нулями"
    passed("1_standardize", "Пропуск 1: стандартизация по train верна")


_check_standardize()

Дальше готовый код: признаки кладём на видеокарту **целиком**, а мини-батч —
это срез по перемешанным индексам. Стандартный `DataLoader(TensorDataset(...))`
собирает батч из отдельных строк циклом на Python: на 400 тыс. строк это
секунды на эпоху вместо долей секунды.

In [ ]:
X_num, NUM_MEAN, NUM_STD = standardize(X_num_raw, idx_train)
DATA = {
    "x": torch.tensor(np.concatenate([np.asarray(X_num, dtype=np.float32), X_onehot], axis=1),
                      device=device),                                   # [N, 10 + 44]
    "y": torch.tensor(y_all, device=device),                            # [N]
}
IDX = {k: torch.tensor(v, device=device)
       for k, v in [("train", idx_train), ("val", idx_val), ("test", idx_test)]}


def iterate_minibatches(idx, batch_size, shuffle=False, generator=None):
    """Отдаёт тензоры индексов строк по batch_size штук."""
    if shuffle:
        idx = idx[torch.randperm(len(idx), device=idx.device, generator=generator)]
    for i in range(0, len(idx), batch_size):
        yield idx[i:i + batch_size]


x_tr = DATA["x"][IDX["train"], :10]
print("числовые признаки после стандартизации, train: среднее %.3f, std %.3f" %
      (x_tr.mean().item(), x_tr.std().item()))

## Пропуск 2. Cross-entropy на логитах — два варианта

Сеть выдаёт **логиты** $a_1, \dots, a_K$ — по числу на класс, без softmax.
Функция потерь для объекта с правильным классом $y$ — минус логарифм
вероятности этого класса. Возвращать нужно **среднее** по батчу — одно число.

`logits` — `[B, K]`, `y` — `[B]` целых номеров классов. Градиент считать не нужно:
всё из операций `torch`, autograd сделает сам. Выбрать из каждой строки элемент
правильного класса поможет `logits[torch.arange(B), y]` или `logits.gather`.

### 2а. В лоб

Сначала softmax, потом логарифм — ровно по определению:

$$
p_k = \frac{e^{a_k}}{\sum_{j=1}^{K} e^{a_j}}, \qquad L = -\log p_y .
$$

In [ ]:
def cross_entropy_naive(logits, y):
    """Средняя по батчу cross-entropy «в лоб»: softmax, затем log. logits — [B, K], y — [B]."""
    # TODO 2а: p = softmax по формуле выше, L = −log p_y; среднее по батчу.
    raise NotImplementedError

In [ ]:
# ПРОВЕРКА 2а — не меняйте эту ячейку
def _check_ce(fn):
    torch.manual_seed(0)
    logits = (torch.randn(64, 7, dtype=torch.float64) * 3).requires_grad_(True)
    y = torch.randint(0, 7, (64,))
    mine = fn(logits, y)
    ref = F.cross_entropy(logits, y)
    assert torch.is_tensor(mine) and mine.dim() == 0, \
        "функция должна вернуть одно число (тензор без размерностей) — среднее по батчу"
    if abs(mine.item() - ref.item() * 64) < 1e-6 * abs(ref.item() * 64):
        raise AssertionError("похоже, вы вернули сумму по батчу, а нужно среднее")
    assert abs(mine.item() - ref.item()) < 1e-6, \
        f"на обычных логитах {mine.item():.6f}, а должно быть {ref.item():.6f}"
    g_mine, = torch.autograd.grad(mine, logits)
    g_ref, = torch.autograd.grad(ref, logits)
    assert torch.allclose(g_mine, g_ref, atol=1e-8), \
        "значение сходится, а градиент нет — где-то оборван граф (.item(), numpy, .data?)"


_check_ce(cross_entropy_naive)
passed("2a_ce_naive", "Пропуск 2а: cross-entropy «в лоб» совпадает с PyTorch")

### 2б. Устойчиво к большим логитам

Вариант «в лоб» ломается на больших логитах: при $a_k = 1000$ значение
$e^{1000}$ не помещается во float и становится `inf`, а при $a_k = -1000$
вероятность округляется до нуля и $\log 0 = -\infty$. Ниже вы это увидите.

Поэтому перепишем $L$ без деления и вынесем из-под экспоненты максимальный логит $m$:

$$
L = -a_y + \log \sum_{k=1}^{K} e^{a_k}
  = m + \log \sum_{k=1}^{K} e^{a_k - m} - a_y,
\qquad m = \max_k a_k .
$$

Максимум $m$ свой у каждой строки батча. После вычитания все показатели
$a_k - m \le 0$, так что экспонента не переполняется, а слагаемое с $a_k = m$ равно
$e^0 = 1$, так что сумма не бывает меньше единицы и логарифм всегда конечен.
Реализуйте $L$ по правой части формулы.

In [ ]:
def cross_entropy(logits, y):
    """Средняя по батчу cross-entropy, устойчивая к большим логитам. logits — [B, K], y — [B]."""
    # TODO 2б: L = m + log Σ exp(a_k − m) − a_y, где m — максимум логитов в строке; среднее по батчу.
    raise NotImplementedError

In [ ]:
# ПРОВЕРКА 2б — не меняйте эту ячейку
def _check_cross_entropy():
    _check_ce(cross_entropy)
    big = torch.tensor([[1000.0, 0.0, -1000.0], [0.0, 1000.0, 1000.0],
                        [-1000.0, -1000.0, -999.0]])
    yb = torch.tensor([2, 2, 0])
    v = cross_entropy(big, yb)
    assert torch.isfinite(v), \
        "на логитах ±1000 получилось inf или nan — вычтите из логитов строки их максимум m"
    want = F.cross_entropy(big, yb).item()
    assert abs(v.item() - want) < 1e-3 * want, \
        f"на логитах ±1000 значение {v.item():.3f}, а должно быть {want:.3f}"
    passed("2_cross_entropy", "Пропуск 2б: cross-entropy совпадает с PyTorch и не боится больших логитов")
    print(f"логиты ±1000: в лоб {cross_entropy_naive(big, yb).item()}, "
          f"устойчиво {v.item():.1f}, PyTorch {want:.1f}")


_check_cross_entropy()

## Пропуск 3. Своя MLP на PyTorch

Сеть для всех трёх частей собираете вы:

```
x [B, d_in] → Linear → ReLU → Linear → ReLU → … → Linear → логиты [B, n_classes]
```

* по скрытому слою `Linear → ReLU` на каждый размер из `hidden`: при
  `hidden = (256, 256, 256)` — три скрытых слоя по 256 нейронов;
  размер входа каждого слоя равен размеру выхода предыдущего;
* выходной `Linear` — **без активации**: сеть возвращает логиты,
  а softmax сидит внутри вашей `cross_entropy`.

Как устроен модуль PyTorch, было на семинаре 1: класс наследуется от `nn.Module`,
в `__init__` создаются слои, в `forward` — прямой ход.

⚠️ Ловушка: PyTorch находит параметры сети, только если слои лежат в атрибутах,
которые сами являются модулями. Обычный список Python `self.layers = [nn.Linear(...), ...]`
он не видит: `model.parameters()` окажется пустым, и оптимизатору нечего будет учить.
Складывайте слои в `nn.Sequential(*layers)` или `nn.ModuleList(layers)`.

Параметр `bn` пока не трогайте — это пропуск 4 в части 2.

In [ ]:
class MLP(nn.Module):
    """Полносвязная сеть для классификации.

    d_in      — число входных признаков;
    hidden    — размеры скрытых слоёв, например (256, 256, 256);
    n_classes — число классов, столько логитов на выходе;
    bn        — BatchNorm в скрытых слоях (пропуск 4, часть 2).
    """
    def __init__(self, d_in, hidden, n_classes, bn=False):
        super().__init__()
        # TODO 3: скрытые слои Linear → ReLU с размерами из hidden, потом выходной Linear.
        # TODO 4 (часть 2): при bn=True скрытый слой — Linear без смещения → BatchNorm1d → ReLU.
        raise NotImplementedError

    def forward(self, x):
        """x — [B, d_in] → логиты [B, n_classes]."""
        # TODO 3
        raise NotImplementedError

In [ ]:
# ПРОВЕРКА 3 — не меняйте эту ячейку
def _check_mlp():
    torch.manual_seed(0)
    net = MLP(4, (5, 7), 3)
    assert isinstance(net, nn.Module), "MLP должна наследоваться от nn.Module"
    n = sum(p.numel() for p in net.parameters())
    assert n > 0, ("у сети нет ни одного параметра: слои лежат в обычном списке Python, "
                   "и PyTorch их не видит. Используйте nn.Sequential или nn.ModuleList")
    lins = [m for m in net.modules() if isinstance(m, nn.Linear)]
    got = [(l.in_features, l.out_features) for l in lins]
    assert got == [(4, 5), (5, 7), (7, 3)], \
        f"линейные слои {got}, а при d_in=4, hidden=(5, 7), n_classes=3 нужны [(4, 5), (5, 7), (7, 3)]"
    want = (4 * 5 + 5) + (5 * 7 + 7) + (7 * 3 + 3)
    assert n == want, f"у сети {n} параметров, а должно быть {want}: у каждого Linear матрица и смещение"
    assert not any(isinstance(m, nn.BatchNorm1d) for m in net.modules()), \
        "при bn=False BatchNorm в сети не нужен"
    x = torch.randn(16, 4)
    out = net(x)
    assert out.shape == (16, 3), f"выход формы {tuple(out.shape)}, а нужна (16, 3)"
    h = x
    for lin in lins[:-1]:
        h = torch.relu(lin(h))
    ref = lins[-1](h)
    assert not torch.allclose(out.sum(1), torch.ones(16), atol=1e-5), \
        "выходы в сумме дают 1 — похоже, на выходе softmax. Нужны логиты: softmax внутри cross_entropy"
    assert not ((out >= 0).all() and not (ref >= 0).all()), \
        "на выходе нет отрицательных чисел — похоже, после выходного Linear стоит ReLU. Нужны логиты"
    assert torch.allclose(out, ref, atol=1e-6), \
        "выход не совпадает с Linear → ReLU → … → Linear: проверьте порядок слоёв и ReLU между ними"
    passed("3_mlp", "Пропуск 3: MLP собрана верно")


_check_mlp()

## Цикл обучения (готов)

Цикл ниже учит вашу `MLP` на вашей `cross_entropy`. Разберитесь, как он устроен:
признаки уже лежат на видеокарте, батч — срез `DATA["x"][b]` по индексам строк.
Цикл ссылается на `MyAdamW` из части 3 — в частях 1–2 он не вызывается.
Темп обучения в частях 2–3 меняется после каждого батча: линейный разогрев
(`LinearLR`), потом косинус до нуля (`CosineAnnealingLR`), склеенные через
`SequentialLR` — то же расписание, что на семинаре 2, только шаг не по эпохам, а по батчам.

In [ ]:
import warnings
# SequentialLR внутри дёргает step(epoch) и сам же на это ругается — предупреждение
# относится к внутренностям PyTorch, а не к нашему коду. Гасим, чтобы не мешало читать.
warnings.filterwarnings("ignore", message=r".*epoch parameter in .scheduler.step.*")


def warmup_cosine(optimizer, total_steps, warmup_steps):
    """Линейный разогрев за warmup_steps шагов, потом косинус до нуля — как на семинаре 2."""
    warm = torch.optim.lr_scheduler.LinearLR(
        optimizer, start_factor=0.01, end_factor=1.0, total_iters=warmup_steps)
    cos = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=total_steps - warmup_steps)
    return torch.optim.lr_scheduler.SequentialLR(optimizer, [warm, cos], milestones=[warmup_steps])


@torch.no_grad()
def evaluate(model, idx, batch_size=16384):
    """Средний loss и accuracy. model.eval() обязателен: BatchNorm."""
    model.eval()
    loss_sum, correct = 0.0, 0
    for b in iterate_minibatches(idx, batch_size):
        out = model(DATA["x"][b])
        y = DATA["y"][b]
        loss_sum += cross_entropy(out, y).item() * len(b)
        correct += (out.argmax(1) == y).sum().item()
    return loss_sum / len(idx), correct / len(idx)


def train(cfg):
    torch.manual_seed(cfg["seed"])
    model = MLP(DATA["x"].shape[1], cfg["hidden"], n_classes=7, bn=cfg["bn"]).to(device)
    if cfg["optimizer"] == "mine":
        optimizer = MyAdamW(model.parameters(), lr=cfg["lr"],
                            weight_decay=cfg["weight_decay"])         # ← пропуск 5, часть 3
    else:
        optimizer = torch.optim.AdamW(model.parameters(), lr=cfg["lr"], weight_decay=0.0)
    steps_per_epoch = math.ceil(len(IDX["train"]) / cfg["batch_size"])
    total = cfg["epochs"] * steps_per_epoch
    scheduler = None
    if cfg["warmup"] is not None:
        scheduler = warmup_cosine(optimizer, total, int(cfg["warmup"] * total))
    gen = torch.Generator(device=device).manual_seed(cfg["seed"])
    hist = {k: [] for k in ["train_loss", "val_loss", "train_acc", "val_acc", "lr"]}
    t0 = time.time()

    for epoch in range(cfg["epochs"]):
        model.train()
        # копим на видеокарте, а не .item() на каждом батче: иначе каждый шаг ждёт GPU
        loss_sum = torch.zeros((), device=device)
        correct = torch.zeros((), device=device)
        for b in iterate_minibatches(IDX["train"], cfg["batch_size"], shuffle=True, generator=gen):
            out = model(DATA["x"][b])
            y = DATA["y"][b]
            loss = cross_entropy(out, y)
            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            optimizer.step()
            if scheduler is not None:
                scheduler.step()
            loss_sum += loss.detach() * len(b)
            correct += (out.argmax(1) == y).sum()

        n = len(IDX["train"])
        hist["train_loss"].append(loss_sum.item() / n)
        hist["train_acc"].append(correct.item() / n)
        hist["lr"].append(optimizer.param_groups[0]["lr"])
        val_loss, val_acc = evaluate(model, IDX["val"])
        hist["val_loss"].append(val_loss)
        hist["val_acc"].append(val_acc)
        if epoch % 5 == 0 or epoch == cfg["epochs"] - 1:
            print(f"эпоха {epoch:3d} | train loss {hist['train_loss'][-1]:.4f} | "
                  f"val loss {val_loss:.4f} | val acc {val_acc:.4f} | {time.time() - t0:5.0f} с")

    print(f"обучение {(time.time() - t0) / 60:.1f} мин")
    return model, hist


def plot_history(hist, title):
    fig, axes = plt.subplots(1, 3, figsize=(16, 3.8))
    for ax, key in zip(axes[:2], ["loss", "acc"]):
        ax.plot(hist[f"train_{key}"], "--", label="train")
        ax.plot(hist[f"val_{key}"], lw=2, label="val")
        ax.set_title(key); ax.set_xlabel("эпоха"); ax.grid(alpha=0.3); ax.legend()
    axes[2].plot(hist["lr"]); axes[2].set_title("lr (конец эпохи)")
    axes[2].set_xlabel("эпоха"); axes[2].grid(alpha=0.3)
    fig.suptitle(title); plt.tight_layout(); plt.show()


def check_quality(part, model):
    """Точность на тесте и сравнение с порогом части."""
    thresholds = {1: 0.90, 2: 0.95, 3: 0.965}
    _, acc = evaluate(model, IDX["test"])
    TEST_ACC[part] = acc
    mark = {1: "«3»", 2: "«4»", 3: "«5»"}[part]
    verdict = "✅ порог пройден" if acc >= thresholds[part] else "❌ ниже порога"
    print(f"Часть {part}: точность на тесте {acc:.4f} (порог на {mark}: {thresholds[part]}) — {verdict}")

## Обучение, часть 1

Ваша MLP 256-256-256 на ваших стандартизованных признаках и вашей функции потерь,
AdamW из PyTorch с постоянным `lr = 1e-3`, 30 эпох. Около минуты на видеокарте.

In [ ]:
P1 = dict(hidden=(256, 256, 256), bn=False, optimizer="torch", lr=1e-3, weight_decay=0.0,
          warmup=None, epochs=30, batch_size=1024, seed=0)
model_p1, hist_p1 = train(P1)
plot_history(hist_p1, "Часть 1")
check_quality(1, model_p1)

---

# Часть 2. BatchNorm → «4»

## Пропуск 4. BatchNorm в вашей MLP

Вернитесь к классу `MLP` (пропуск 3) и допишите вариант `bn=True`. Каждый скрытый
слой тогда такой:

```
Linear → BatchNorm1d → ReLU
```

Нормализация — между линейным слоем и нелинейностью, как в `make_mlp` на семинаре.
Выходной слой — прежний `Linear`, без нормализации.

И одна деталь с лекции: **перед BatchNorm смещение у линейного слоя не нужно**.
Подумайте, что BatchNorm делает с постоянным сдвигом `+ b`, который линейный слой
добавляет ко всем объектам батча, — и чем его заменяет.

После правки перезапустите ячейку с классом, а потом проверку ниже.

In [ ]:
# ПРОВЕРКА 4 — не меняйте эту ячейку
def _check_mlp_bn():
    torch.manual_seed(0)
    net = MLP(4, (5, 7), 3, bn=True)
    lins = [m for m in net.modules() if isinstance(m, nn.Linear)]
    bns = [m for m in net.modules() if isinstance(m, nn.BatchNorm1d)]
    assert [(l.in_features, l.out_features) for l in lins] == [(4, 5), (5, 7), (7, 3)], \
        "линейные слои при bn=True должны быть те же, что без BatchNorm"
    assert len(bns) == 2, f"BatchNorm1d в сети {len(bns)} шт., а нужен в каждом скрытом слое — 2"
    assert [b.num_features for b in bns] == [5, 7], \
        "BatchNorm1d нормирует выходы своего Linear: здесь 5 и 7 признаков"
    assert lins[0].bias is None and lins[1].bias is None, \
        "перед BatchNorm у Linear не должно быть смещения"
    assert lins[2].bias is not None, "у выходного Linear смещение нужно: BatchNorm после него нет"
    n = sum(p.numel() for p in net.parameters())
    want = (4 * 5 + 2 * 5) + (5 * 7 + 2 * 7) + (7 * 3 + 3)
    assert n == want, f"у сети {n} параметров, а должно быть {want}"
    x = torch.randn(32, 4)
    for mode in ("train", "eval"):
        net.train(mode == "train")
        out = net(x)
        h = x
        for lin, bn in zip(lins[:-1], bns):
            h = torch.relu(bn(lin(h)))
        ref = lins[-1](h)
        assert out.shape == (32, 3), f"выход формы {tuple(out.shape)}, а нужна (32, 3)"
        assert torch.allclose(out, ref, atol=1e-5), \
            f"в режиме {mode} выход не совпадает с Linear → BatchNorm1d → ReLU → … → Linear"
    passed("4_mlp_bn", "Пропуск 4: BatchNorm в MLP стоит верно")


_check_mlp_bn()

## Обучение, часть 2

То же, что в части 1, плюс BatchNorm в каждом скрытом слое и темп побольше
(`3e-3`) с расписанием: разогрев на 5% шагов, потом косинус до нуля.

In [ ]:
P2 = dict(P1, bn=True, lr=3e-3, warmup=0.05)
model_p2, hist_p2 = train(P2)
plot_history(hist_p2, "Часть 2")
check_quality(2, model_p2)

---

# Часть 3. Свой AdamW и финальная модель → «5»

## Пропуск 5. Adam с L2 против AdamW

На лекции и на семинаре: `Adam(weight_decay=…)` и `AdamW(weight_decay=…)` —
**разные алгоритмы**. Здесь вы напишете один шаг обоих и сверите с PyTorch,
а потом финальная модель будет учиться на вашем AdamW.

Шаг Adam для одного тензора параметров `p` с градиентом `g` на шаге `t` (счёт с 1):

```
m ← β₁·m + (1 − β₁)·g              v ← β₂·v + (1 − β₂)·g²
m̂ = m / (1 − β₁ᵗ)                   v̂ = v / (1 − β₂ᵗ)
p ← p − lr · m̂ / (√v̂ + ε)
```

Куда входит weight decay `λ`:

* **Adam с L2** (`decoupled=False`): штраф `λ/2·‖p‖²` добавляется к функции
  потерь, то есть `λ·p` добавляется **к градиенту** до всех формул выше;
* **AdamW** (`decoupled=True`): градиент не трогаем, а веса **отдельно** сжимаются
  в `(1 − lr·λ)` раз — перед шагом Adam.

Функция ничего не меняет на месте, а возвращает новые `p, m, v`.

In [ ]:
def adam_step(p, g, m, v, t, lr, weight_decay, decoupled, betas=(0.9, 0.999), eps=1e-8):
    b1, b2 = betas
    # TODO 5: один шаг Adam; weight decay — как L2 (decoupled=False) или как в AdamW.
    raise NotImplementedError

In [ ]:
# ПРОВЕРКА 5 — не меняйте эту ячейку
def _check_adam_step():
    for decoupled, Opt in [(False, torch.optim.Adam), (True, torch.optim.AdamW)]:
        name = "AdamW" if decoupled else "Adam с L2"
        torch.manual_seed(0)
        p0 = torch.randn(5, 4, dtype=torch.float64)
        grads = [torch.randn(5, 4, dtype=torch.float64) * s for s in (1.0, 0.1, 3.0, 0.5, 2.0)]
        ref = p0.clone().requires_grad_(True)
        opt = Opt([ref], lr=0.01, weight_decay=0.3, betas=(0.9, 0.999), eps=1e-8)
        p, m, v = p0.clone(), torch.zeros_like(p0), torch.zeros_like(p0)
        for t, g in enumerate(grads, start=1):
            ref.grad = g.clone(); opt.step()
            p, m, v = adam_step(p, g.clone(), m, v, t, lr=0.01, weight_decay=0.3,
                                decoupled=decoupled)
            err = (p - ref.detach()).abs().max().item()
            assert err < 1e-5, (f"{name}: после шага {t} веса расходятся с torch.optim."
                                f"{Opt.__name__} на {err:.1e}. Проверьте, куда и в какой момент "
                                f"входит weight decay (см. описание выше) и поправку "
                                f"на смещение моментов")
    z = torch.zeros_like(p0)
    a, _, _ = adam_step(p0, grads[0], z, z, 1, 0.01, 0.3, False)
    b, _, _ = adam_step(p0, grads[0], z, z, 1, 0.01, 0.3, True)
    assert (a - b).abs().max() > 1e-6, "Adam с L2 и AdamW дали одинаковый шаг — так не бывает"
    passed("5_adam_step", "Пропуск 5: шаги Adam с L2 и AdamW совпадают с PyTorch")


_check_adam_step()

Чем эта разница оборачивается на практике — ваша функция на двух группах весов
по тысяче штук. Все веса стартуют с 1.0, weight decay у всех одинаковый (λ = 1),
`lr = 1e-3`, 1000 шагов. Градиент от функции потерь — чистый шум: у первой группы
маленький (σ = 0.001), у второй большой (σ = 10). Шум в среднем ни к чему
не тянет, поэтому всё, на сколько веса уменьшились, — работа weight decay.

In [ ]:
torch.manual_seed(0)
scale = torch.tensor([[1e-3], [10.0]], dtype=torch.float64)       # σ шума для двух групп
print(f"{'':>10} | {'малый градиент':>15} | {'большой градиент':>16}   (средний вес после 1000 шагов)")
for decoupled, name in [(False, "Adam + L2"), (True, "AdamW")]:
    p = torch.ones(2, 1000, dtype=torch.float64)
    m, v = torch.zeros_like(p), torch.zeros_like(p)
    for t in range(1, 1001):
        g = torch.randn(2, 1000, dtype=torch.float64) * scale
        p, m, v = adam_step(p, g, m, v, t, lr=1e-3, weight_decay=1.0, decoupled=decoupled)
    print(f"{name:>10} | {p[0].mean().item():15.3f} | {p[1].mean().item():16.3f}")
print(f"для сравнения, (1 − lr·λ)^1000 = {(1 - 1e-3) ** 1000:.3f}")

У AdamW decay одинаковый для обеих групп и ровно такой, как обещает формула:
`(1 − lr·λ)¹⁰⁰⁰`. У Adam с L2 штраф `λ·p` проходит через деление на `√v̂` — и всё
зависит от масштаба градиента: веса с большим градиентом decay почти не трогает,
веса с маленьким прижимает сильнее, чем AdamW. То есть L2 внутри Adam штрафует
**не те веса**, которые вы собирались штрафовать. Поэтому с Adam берут AdamW.

## Оптимизатор на вашем шаге (готов)

`MyAdamW` — обёртка, которая для каждого параметра хранит `m`, `v`, `t` и вызывает
ваш `adam_step` с `decoupled=True`. Weight decay достаётся только матрицам весов:
смещения и параметры BatchNorm (одномерные тензоры) от него освобождены — они
не отвечают за «сложность» функции, а только сдвигают и масштабируют.

In [ ]:
class MyAdamW(torch.optim.Optimizer):
    def __init__(self, params, lr, weight_decay):
        super().__init__(params, dict(lr=lr, weight_decay=weight_decay))

    @torch.no_grad()
    def step(self):
        for group in self.param_groups:
            for p in group["params"]:
                if p.grad is None:
                    continue
                st = self.state[p]
                if not st:
                    st["t"], st["m"], st["v"] = 0, torch.zeros_like(p), torch.zeros_like(p)
                st["t"] += 1
                wd = group["weight_decay"] if p.dim() >= 2 else 0.0
                new_p, st["m"], st["v"] = adam_step(p, p.grad, st["m"], st["v"], st["t"],
                                                    group["lr"], wd, decoupled=True)
                p.copy_(new_p)

## Обучение, часть 3: финальная модель

Сеть шире и глубже (512×4), 60 эпох, ваш AdamW с weight decay 0.01. Батч вдвое
больше (2048): шагов на эпоху вдвое меньше, поэтому вдвое больше эпох обходится
заметно меньше чем вдвое дороже по времени; темп обучения подняли до `5e-3`.

In [ ]:
P3 = dict(P2, hidden=(512, 512, 512, 512), optimizer="mine", lr=5e-3, weight_decay=0.01,
          epochs=60, batch_size=2048)
model_p3, hist_p3 = train(P3)
plot_history(hist_p3, "Часть 3: финальная модель")
check_quality(3, model_p3)

---

# Итог

Ячейка ниже собирает всё: какие проверки пройдены и какая точность на тесте
в каждой части. Выполняйте ноутбук **целиком сверху вниз** — итог учитывает
только то, что реально выполнилось в этом запуске.

In [ ]:
PARTS = {1: ["1_standardize", "2a_ce_naive", "2_cross_entropy", "3_mlp"],
         2: ["4_mlp_bn"],
         3: ["5_adam_step"]}
THRESHOLDS = {1: 0.90, 2: 0.95, 3: 0.965}
grade, rows = 2, []
for part in (1, 2, 3):
    checks_ok = all(PASSED.get(c, False) for c in PARTS[part])
    acc = TEST_ACC.get(part, float("nan"))
    ok = checks_ok and acc >= THRESHOLDS[part]
    rows.append({"часть": part, "проверки": "все ✅" if checks_ok else
                 "не пройдены: " + ", ".join(c for c in PARTS[part] if not PASSED.get(c)),
                 "тест": round(acc, 4), "порог": THRESHOLDS[part], "часть засчитана": ok})
    if ok and grade == part + 1:
        grade = part + 2
display(pd.DataFrame(rows).set_index("часть"))
print("ОЦЕНКА:", grade if grade >= 3 else "2 (не засчитана часть 1)")
print(f"весь ноутбук: {(time.time() - T_START) / 60:.1f} мин")